In [1]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
def load_audio(path, sr=22050, duration=5):
    audio, _ = librosa.load(path, sr=sr)
    max_len = sr * duration
    if len(audio) < max_len:
        audio = np.pad(audio, (0, max_len - len(audio)))
    return audio[:max_len]


def random_tempo(audio, sr=22050, duration=5):
    rate = random.uniform(0.95, 1.05)
    audio = librosa.effects.time_stretch(audio, rate=rate)

    max_len = sr * duration
    if len(audio) < max_len:
        audio = np.pad(audio, (0, max_len - len(audio)))
    return audio[:max_len]



def add_noise(audio, noise, snr_db):
    audio_power = np.mean(audio**2)
    noise_power = np.mean(noise**2) + 1e-8
    factor = np.sqrt(audio_power / (10**(snr_db/10) * noise_power))
    return audio + factor * noise[:len(audio)]


def audio_to_mel(audio, sr=22050):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=128,
        n_fft=2048,
        hop_length=512
    )
    mel = librosa.power_to_db(mel)

    # Normalize per sample
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    return mel.astype(np.float32)

In [4]:
class MashupDataset(Dataset):
    def __init__(self, root_dir, noise_dir, song_dict, genres):
        self.root_dir = root_dir
        self.noise_files = [
            os.path.join(noise_dir, f)
            for f in os.listdir(noise_dir)
            if f.endswith(".wav")
        ]
        self.song_dict = song_dict
        self.genres = genres
        self.genre_to_idx = {g:i for i,g in enumerate(genres)}

        self.length = 3000

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        genre = random.choice(self.genres)
        songs = self.song_dict[genre]

        mixed = 0
        max_len = 22050 * 5

        for stem_name in ["drums.wav", "bass.wav", "vocals.wav", "other.wav"]:
            song = random.choice(songs)
            stem_path = os.path.join(self.root_dir, genre, song, stem_name)

            stem = load_audio(stem_path)

            # Tempo shift
            rate = random.uniform(0.95, 1.05)
            stem = librosa.effects.time_stretch(stem, rate=rate)

            # Gain variation
            stem *= random.uniform(0.7, 1.3)

            # Fix length
            if len(stem) < max_len:
                stem = np.pad(stem, (0, max_len - len(stem)))
            stem = stem[:max_len]

            mixed += stem

        # Add noise
        noise_path = random.choice(self.noise_files)
        noise = load_audio(noise_path)
        mixed = add_noise(mixed, noise, random.randint(5, 20))

        mel = audio_to_mel(mixed)
        mel = torch.tensor(mel).unsqueeze(0)

        label = self.genre_to_idx[genre]
        return mel, label

In [5]:
def prepare_song_split(root_dir, val_ratio=0.2):
    genres = sorted(os.listdir(root_dir))
    train_songs = {}
    val_songs = {}

    for genre in genres:
        genre_path = os.path.join(root_dir, genre)
        songs = os.listdir(genre_path)

        train, val = train_test_split(
            songs,
            test_size=val_ratio,
            random_state=42
        )

        train_songs[genre] = train
        val_songs[genre] = val

    return train_songs, val_songs, genres